# 04 · Synthetic Driver Data Generation

Generates ~5 000 synthetic driver records that **preserve inter-feature correlations**
from `processed/driver_features.csv`, then combines them with the original data and
writes `processed/final_driver_dataset.csv`.

### Strategy
* **Primary** – [CTGAN](https://sdv.dev/SDV/user_guides/single_table/ctgan.html) from the
  `sdv` library.  CTGAN learns the joint distribution (including correlations) via a
  conditional GAN and natively handles mixed categorical / numerical columns.
* **Fallback** – Gaussian Copula (also from `sdv`) if CTGAN is not available.

Both paths honour `city`, `shift_preference`, and `risk_label` as discrete columns.

## 0 · Install dependencies (run once)

In [ ]:
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("sdv")
_ensure("pandas")
_ensure("numpy")
_ensure("matplotlib")
_ensure("scipy")
print("All dependencies ready.")

## 1 · Imports & configuration

In [ ]:
import warnings, pathlib, importlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# ── paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = pathlib.Path("../")
INPUT_CSV   = BASE_DIR / "processed" / "driver_features.csv"
OUTPUT_CSV  = BASE_DIR / "processed" / "final_driver_dataset.csv"

TARGET_SYNTHETIC = 5_000
RANDOM_SEED      = 42

np.random.seed(RANDOM_SEED)
print(f"Input  : {INPUT_CSV}")
print(f"Output : {OUTPUT_CSV}")
print(f"Target synthetic rows: {TARGET_SYNTHETIC:,}")

## 2 · Load & inspect the original data

In [ ]:
df_orig = pd.read_csv(INPUT_CSV)

FEATURE_COLS = [
    "city", "shift_preference",
    "avg_hours_per_day", "avg_earnings_per_hour",
    "experience_months", "rating", "daily_productivity",
    "avg_combined_score", "avg_motion_score", "avg_audio_score",
    "total_flags", "risk_score", "risk_label",
]

CAT_COLS = ["city", "shift_preference", "risk_label"]
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

print(f"Original shape : {df_orig.shape}")
print(f"\nNumeric columns  : {NUM_COLS}")
print(f"Categorical cols : {CAT_COLS}")
df_orig[FEATURE_COLS].head()

## 3 · Define SDV metadata

In [ ]:
from sdv.metadata import SingleTableMetadata

working_df = df_orig[FEATURE_COLS].copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(working_df)

# Explicitly tag categorical columns so the model handles them correctly
for col in CAT_COLS:
    metadata.update_column(column_name=col, sdtype="categorical")

print("Metadata summary:")
metadata.visualize()

## 4 · Train the generative model (CTGAN → Gaussian Copula fallback)

In [ ]:
synthesizer = None
method_used = None

# ── Try CTGAN first ────────────────────────────────────────────────────────────
try:
    from sdv.single_table import CTGANSynthesizer
    print("▶ Training CTGAN …  (this may take 1-2 min)")
    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=500,
        batch_size=50,      # small dataset → smaller batch
        generator_dim=(256, 256),
        discriminator_dim=(256, 256),
        verbose=True,
        cuda=False,
    )
    synthesizer.fit(working_df)
    method_used = "CTGAN"
    print("\n✅ CTGAN training complete.")
except Exception as e:
    print(f"CTGAN unavailable ({e}).  Falling back to Gaussian Copula …")

# ── Fallback: Gaussian Copula ──────────────────────────────────────────────────
if synthesizer is None:
    from sdv.single_table import GaussianCopulaSynthesizer
    print("▶ Training Gaussian Copula …")
    synthesizer = GaussianCopulaSynthesizer(
        metadata,
        default_distribution="beta",
    )
    synthesizer.fit(working_df)
    method_used = "Gaussian Copula"
    print("✅ Gaussian Copula training complete.")

print(f"\nModel used: {method_used}")

## 5 · Sample synthetic records

In [ ]:
df_synthetic = synthesizer.sample(num_rows=TARGET_SYNTHETIC, batch_size=500)

# ── Post-processing: clip numeric columns to realistic ranges ─────────────────
clip_ranges = {
    "avg_hours_per_day"     : (df_orig["avg_hours_per_day"].min(),      df_orig["avg_hours_per_day"].max()),
    "avg_earnings_per_hour" : (df_orig["avg_earnings_per_hour"].min(),   df_orig["avg_earnings_per_hour"].max()),
    "experience_months"     : (df_orig["experience_months"].min(),        df_orig["experience_months"].max()),
    "rating"                : (1.0,  5.0),
    "daily_productivity"    : (0.0,  None),
    "avg_combined_score"    : (0.0,  1.0),
    "avg_motion_score"      : (0.0,  1.0),
    "avg_audio_score"       : (0.0,  1.0),
    "total_flags"           : (0.0,  None),
    "risk_score"            : (0.0,  None),
}
for col, (lo, hi) in clip_ranges.items():
    if col in df_synthetic.columns:
        df_synthetic[col] = df_synthetic[col].clip(lower=lo, upper=hi)

# Round integer-like columns
for col in ["avg_earnings_per_hour", "experience_months", "total_flags"]:
    if col in df_synthetic.columns:
        df_synthetic[col] = df_synthetic[col].round().astype(int)

# Derive risk_label from risk_score if the model produced inconsistencies
def _risk_label(score):
    if score >= 30:  return "HIGH"
    elif score >= 20: return "MEDIUM"
    else:            return "LOW"

df_synthetic["risk_label"] = df_synthetic["risk_score"].apply(_risk_label)

# Tag the source
df_synthetic["source"] = "synthetic"

print(f"Synthetic rows generated : {len(df_synthetic):,}")
df_synthetic.head()

## 6 · Combine original + synthetic

In [ ]:
df_orig_tagged = df_orig.copy()
df_orig_tagged["source"] = "original"

# Align columns: keep only FEATURE_COLS + source
keep_cols = FEATURE_COLS + ["source"]
df_combined = pd.concat(
    [df_orig_tagged[keep_cols], df_synthetic[keep_cols]],
    ignore_index=True,
)

# ── Summary counts ─────────────────────────────────────────────────────────────
n_orig      = len(df_orig_tagged)
n_synthetic = len(df_synthetic)
n_final     = len(df_combined)

print("=" * 50)
print(f"  Original row count  : {n_orig:,}")
print(f"  Synthetic row count : {n_synthetic:,}")
print(f"  Final row count     : {n_final:,}")
print("=" * 50)

print("\n── Risk label distribution (final) ─────────────")
rl_dist = df_combined["risk_label"].value_counts()
print(rl_dist.to_string())
print(f"  (proportions)")
print((rl_dist / rl_dist.sum()).round(3).to_string())

print("\n── City distribution (final) ───────────────────")
city_dist = df_combined["city"].value_counts()
print(city_dist.to_string())

## 7 · Summary statistics comparison

In [ ]:
desc_orig = df_orig[NUM_COLS].describe().T
desc_synt = df_synthetic[NUM_COLS].describe().T

desc_orig.columns = [f"orig_{c}" for c in desc_orig.columns]
desc_synt.columns = [f"synt_{c}" for c in desc_synt.columns]

comparison = pd.concat([desc_orig, desc_synt], axis=1)

# Reorder: orig_mean | synt_mean | orig_std | synt_std …
pairs = []
for stat in ["mean", "std", "min", "50%", "max"]:
    pairs += [f"orig_{stat}", f"synt_{stat}"]
existing_pairs = [c for c in pairs if c in comparison.columns]
comparison = comparison[existing_pairs]

print("\n── Summary statistics: Original vs Synthetic ───")
print(comparison.round(3).to_string())

## 8 · Visual comparison

In [ ]:
plot_cols = [
    "avg_hours_per_day", "avg_earnings_per_hour", "experience_months",
    "rating", "daily_productivity", "avg_combined_score",
    "avg_motion_score", "avg_audio_score", "total_flags",
    "risk_score",
]

ncols = 4
nrows = -(-len(plot_cols) // ncols)   # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
fig.suptitle("Distribution Comparison: Original vs Synthetic", fontsize=15, fontweight="bold")

for ax, col in zip(axes.flat, plot_cols):
    bins = 30
    orig_vals = df_orig[col].dropna()
    synt_vals = df_synthetic[col].dropna()

    ax.hist(orig_vals, bins=bins, density=True, alpha=0.55,
            color="steelblue", label="Original")
    ax.hist(synt_vals, bins=bins, density=True, alpha=0.55,
            color="tomato",    label="Synthetic")

    # KS statistic
    ks_stat, ks_p = stats.ks_2samp(orig_vals, synt_vals)
    ax.set_title(f"{col}\nKS={ks_stat:.3f}  p={ks_p:.3f}", fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

# Hide empty subplots
for ax in axes.flat[len(plot_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(BASE_DIR / "processed" / "distribution_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: processed/distribution_comparison.png")

### Categorical column distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Categorical Distribution: Original vs Synthetic", fontsize=13, fontweight="bold")

for ax, col in zip(axes, CAT_COLS):
    orig_pct = df_orig[col].value_counts(normalize=True).sort_index()
    synt_pct = df_synthetic[col].value_counts(normalize=True).sort_index()
    cats = sorted(set(orig_pct.index) | set(synt_pct.index))
    x = range(len(cats))
    w = 0.35
    ax.bar([i - w/2 for i in x], [orig_pct.get(c, 0) for c in cats],
           width=w, label="Original", color="steelblue", alpha=0.8)
    ax.bar([i + w/2 for i in x], [synt_pct.get(c, 0) for c in cats],
           width=w, label="Synthetic", color="tomato",    alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(cats, rotation=30, ha="right", fontsize=8)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("Proportion")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(BASE_DIR / "processed" / "categorical_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: processed/categorical_comparison.png")

### Correlation heatmaps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

def _plot_corr(df, ax, title):
    corr = df[NUM_COLS].corr()
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(len(NUM_COLS)))
    ax.set_yticks(range(len(NUM_COLS)))
    ax.set_xticklabels(NUM_COLS, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(NUM_COLS, fontsize=8)
    for i in range(len(NUM_COLS)):
        for j in range(len(NUM_COLS)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}",
                    ha="center", va="center", fontsize=6,
                    color="black" if abs(corr.iloc[i, j]) < 0.7 else "white")
    ax.set_title(title, fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

_plot_corr(df_orig,      ax1, "Original – Correlation Matrix")
_plot_corr(df_synthetic, ax2, "Synthetic – Correlation Matrix")

plt.tight_layout()
plt.savefig(BASE_DIR / "processed" / "correlation_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: processed/correlation_comparison.png")

## 9 · Save final dataset

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_combined.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Final dataset saved to: {OUTPUT_CSV}")
print(f"   Rows  : {len(df_combined):,}")
print(f"   Cols  : {len(df_combined.columns)}")
print(f"   Columns: {list(df_combined.columns)}")
df_combined.head()

## 10 · SDV Quality Report (optional)

In [ ]:
try:
    from sdv.evaluation.single_table import run_diagnostic, evaluate_quality

    print("Running SDV diagnostic …")
    diagnostic = run_diagnostic(
        real_data=working_df,
        synthetic_data=df_synthetic[FEATURE_COLS],
        metadata=metadata,
    )
    print("\nRunning SDV quality report …")
    quality_report = evaluate_quality(
        real_data=working_df,
        synthetic_data=df_synthetic[FEATURE_COLS],
        metadata=metadata,
    )
    print(f"\n Overall Quality Score: {quality_report.get_score():.4f}")
except ImportError:
    print("SDV evaluation module not found – skipping quality report.")
except Exception as e:
    print(f"Quality report skipped: {e}")